<a href="https://colab.research.google.com/github/anatormin/Estrutura-De-Dados-II/blob/main/Aula_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ==============================================================================
# SISTEMA DE ATENDIMENTO DA CLÍNICA - PRIORIDADE PARA AUTISMO
# ==============================================================================
#
# OBJETIVO GERAL:
# Gerenciar uma fila de atendimento onde pacientes com autismo têm prioridade
# sobre pacientes sem autismo.
#
# COMO FUNCIONA:
# - Pacientes com autismo (autismo = True) entram na frente dos demais
# - Entre pacientes com autismo, respeita-se a ordem de chegada
# - Pacientes sem autismo vão para o final da fila (ordem de chegada)
#
# ESTRUTURA UTILIZADA:
# - Lista encadeada simples (cada nó aponta para o próximo)
# - Mantemos referências para o início (primeiro) e fim (último) da fila
# ==============================================================================


# ==============================================================================
# CLASSE 1: Paciente
# ==============================================================================
# Responsabilidade: Armazenar os dados de uma pessoa que será atendida.
# ==============================================================================

class Paciente:
    """
    Representa um paciente com suas informações pessoais e status de prioridade.
    """

    def __init__(self, nome: str, idade: int, autismo: bool = False):
        """
        Construtor da classe Paciente.

        PARÂMETROS:
        - nome (str): Nome completo do paciente
        - idade (int): Idade em anos
        - autismo (bool): True se a pessoa tem autismo, False caso contrário
                          (padrão é False, ou seja, sem autismo)

        ATRIBUTOS CRIADOS:
        - self.nome: armazena o nome
        - self.idade: armazena a idade
        - self.autismo: armazena o status de autismo
        - self.eh_prioritario: define se o paciente tem direito à prioridade
                               (neste caso, autismo = prioridade)
        """
        self.nome = nome
        self.idade = idade
        self.autismo = autismo
        # A prioridade é definida pelo autismo, não pela idade!
        # Isso torna o sistema mais inclusivo e adaptável
        self.eh_prioritario = autismo

    def __repr__(self):
        """
        Método especial do Python.
        Define como o objeto Paciente será EXIBIDO quando usarmos print().

        Exemplo de saída:
        "[Carlos, 18 anos, Autista: Sim - PRIORITÁRIO]"

        Isso facilita a visualização dos dados durante os testes.
        """
        # Converte o booleano autismo para "Sim" ou "Não" (mais amigável)
        status_autismo = "Sim" if self.autismo else "Não"
        # Converte o booleano prioritario para texto maiúsculo (destaca na tela)
        status_prioridade = "PRIORITÁRIO" if self.eh_prioritario else "NORMAL"

        # Retorna uma string formatada com colchetes para destacar cada paciente
        return f"[{self.nome}, {self.idade} anos, Autista: {status_autismo} - {status_prioridade}]"


# ==============================================================================
# CLASSE 2: Node (Nó)
# ==============================================================================
# Responsabilidade: Criar os "elos" da corrente que forma a fila.
# Cada nó guarda um dado (Paciente) e uma referência para o próximo nó.
# ==============================================================================

class Node:
    """
    Representa um elemento individual da lista encadeada.

    A LISTA ENCADEADA é formada por vários nós conectados entre si.
    Cada nó contém:
    1. Um DADO (o Paciente)
    2. Uma REFERÊNCIA (proximo) que aponta para o próximo nó da fila

    O ÚLTIMO nó da fila tem proximo = None (indica o fim da fila).
    """

    def __init__(self, paciente: Paciente):
        """
        Construtor da classe Node.

        PARÂMETRO:
        - paciente (Paciente): O objeto Paciente que será armazenado neste nó

        ATRIBUTOS CRIADOS:
        - self.dado: guarda o paciente (o "conteúdo" do nó)
        - self.proximo: guarda a referência para o próximo nó
                        Inicialmente None porque o nó ainda não foi conectado
        """
        # O dado armazenado é um objeto da classe Paciente (inteiro, não só um nome)
        self.dado = paciente

        # A referência 'proximo' é o que torna a estrutura DINÂMICA!
        # Ela pode apontar para qualquer nó, permitindo crescimento/remoção flexíveis
        self.proximo = None


# ==============================================================================
# CLASSE 3: FilaClinica
# ==============================================================================
# Responsabilidade: Gerenciar a fila de pacientes como um todo.
# Mantém o CONTROLE do início e do fim da fila através de referências.
# ==============================================================================

class FilaClinica:
    """
    Gerencia a fila de atendimento da clínica.

    A FILA é uma estrutura do tipo FIFO (First In, First Out):
    - O primeiro que entra é o primeiro que sai (atendido)

    Porém, com a REGRA DE PRIORIDADE:
    - Pacientes com autismo "furam" a fila e vão para a frente
    - Mas respeitam a ordem de chegada entre si

    ATRIBUTOS PRINCIPAIS:
    - self.inicio: referência para o PRIMEIRO nó da fila (quem será atendido)
    - self.fim: referência para o ÚLTIMO nó da fila (onde novos pacientes normais entram)
    - self._tamanho: contador de quantos pacientes estão na fila
    """

    def __init__(self):
        """
        Construtor da classe FilaClinica.

        Inicializa a fila como VAZIA:
        - inicio = None (não há primeiro paciente)
        - fim = None (não há último paciente)
        - _tamanho = 0 (zero pacientes aguardando)
        """
        # 'inicio' aponta para o nó que está na FRENTE da fila (primeiro a ser atendido)
        # Quando a fila está vazia, inicio = None
        self.inicio = None

        # 'fim' aponta para o nó que está no FINAL da fila (último a ser atendido)
        # Quando a fila está vazia, fim = None
        self.fim = None

        # '_tamanho' é um contador privado (underline indica que é interno)
        # Armazena a quantidade total de pacientes na fila
        self._tamanho = 0

    # ==========================================================================
    # MÉTODO: esta_vazia()
    # ==========================================================================
    # Verifica se a fila tem pacientes ou está vazia.
    # Retorna True se vazia, False se tem pelo menos um paciente.
    # ==========================================================================

    def esta_vazia(self) -> bool:
        """
        Verifica se a fila está vazia.

        LÓGICA: Se o 'inicio' é None, não há nenhum nó na fila.
        Isso acontece porque:
        - Quando adicionamos o primeiro nó, 'inicio' deixa de ser None
        - Quando atendemos o último nó, 'inicio' volta a ser None

        RETORNO:
        - True: fila vazia (não há pacientes)
        - False: fila tem pelo menos um paciente
        """
        # Se não há referência para o início, a fila está vazia
        return self.inicio is None

    # ==========================================================================
    # MÉTODO: adicionar()
    # ==========================================================================
    # Insere um novo paciente na fila, respeitando a regra de prioridade.
    # É a função mais IMPORTANTE do sistema!
    # ==========================================================================

    def adicionar(self, nome: str, idade: int, autismo: bool = False):
        """
        Adiciona um paciente à fila com base na prioridade.

        PARÂMETROS:
        - nome (str): Nome do paciente
        - idade (int): Idade do paciente
        - autismo (bool): True se tem autismo, False se não tem (padrão False)

        REGRA DE INSERÇÃO:
        1. Fila VAZIA → o paciente vira o único da fila (início = fim = novo nó)
        2. Paciente com AUTISMO → insere na frente dos normais, mas depois dos autistas já existentes
        3. Paciente SEM AUTISMO → insere sempre no FINAL da fila

        POR QUE ESSA REGRA?
        - Autistas têm PRIORIDADE sobre os normais
        - Entre autistas, quem chegou primeiro é atendido primeiro (ordem de chegada)
        - Normais esperam sua vez, na ordem de chegada
        """

        # --- PASSO 1: Criar o paciente e o nó ---
        # Primeiro, criamos o objeto Paciente com os dados fornecidos
        novo_paciente = Paciente(nome, idade, autismo)

        # Depois, criamos o nó que vai armazenar esse paciente na fila
        novo_no = Node(novo_paciente)

        # ======================================================================
        # CASO 1: FILA VAZIA
        # ======================================================================
        # Se a fila está vazia (inicio = None), o novo nó será o primeiro e o último
        # Isso é o caso mais simples: não há ninguém na frente nem atrás
        # ======================================================================
        if self.esta_vazia():
            # O novo nó é o início da fila (primeiro a ser atendido)
            self.inicio = novo_no

            # O novo nó também é o fim da fila (último a ser atendido)
            self.fim = novo_no

            # Incrementa o contador de pacientes
            self._tamanho += 1

            # Mensagem de confirmação
            print(f"-> Paciente {novo_paciente.nome} adicionado como o primeiro da fila.")
            return  # Sai da função (não precisa verificar os outros casos)

        # ======================================================================
        # CASO 2: PACIENTE PRIORITÁRIO (tem autismo)
        # ======================================================================
        # O paciente tem direito à prioridade! Precisamos colocá-lo na frente
        # dos pacientes normais, mas respeitando a ordem entre autistas.
        # ======================================================================
        if novo_paciente.eh_prioritario:

            # --- Subcaso 2.1: O primeiro da fila NÃO é prioritário ---
            # Se o início da fila é um paciente NORMAL, o novo prioritário
            # deve entrar na FRENTE dele.
            # ============================================================
            if not self.inicio.dado.eh_prioritario:
                # O novo nó aponta para o antigo início (que era normal)
                # Agora o novo nó está na frente da fila
                novo_no.proximo = self.inicio

                # O início da fila passa a ser o novo nó (prioritário)
                self.inicio = novo_no

            # --- Subcaso 2.2: Já existem pacientes prioritários na fila ---
            # Precisamos encontrar o ÚLTIMO paciente prioritário da fila
            # para inserir o novo DEPOIS dele (respeitando ordem de chegada)
            # ============================================================
            else:
                # 'atual' é uma variável que vamos usar para percorrer a fila
                # Começamos pelo início (que é prioritário)
                atual = self.inicio

                # Enquanto houver um próximo nó E esse próximo for prioritário...
                # A condição do while verifica DUAS coisas:
                # 1. atual.proximo is not None → existe um próximo nó?
                # 2. atual.proximo.dado.eh_prioritario → esse próximo é prioritário?
                # Se ambas forem True, continuamos andando.
                # Paramos quando encontramos o ÚLTIMO prioritário.
                while atual.proximo is not None and atual.proximo.dado.eh_prioritario:
                    # Avança para o próximo nó (caminha na fila)
                    atual = atual.proximo

                # Neste ponto, 'atual' é o ÚLTIMO paciente prioritário da fila
                # O novo nó (prioritário) deve ser inserido DEPOIS dele

                # O novo nó aponta para o nó que estava depois do último prioritário
                # (pode ser um normal ou None, se for o fim da fila)
                novo_no.proximo = atual.proximo

                # O último prioritário agora aponta para o novo nó
                atual.proximo = novo_no

                # Se o novo nó foi inserido no FINAL da fila (não tinha ninguém depois),
                # precisamos atualizar a referência 'fim' para apontar para ele
                if novo_no.proximo is None:
                    self.fim = novo_no

        # ======================================================================
        # CASO 3: PACIENTE NÃO PRIORITÁRIO (não tem autismo)
        # ======================================================================
        # O paciente NÃO tem prioridade. Deve ir para o FINAL da fila,
        # depois de todos os pacientes já existentes.
        # ======================================================================
        else:
            # O antigo fim agora aponta para o novo nó
            # Isso conecta o novo nó ao final da fila
            self.fim.proximo = novo_no

            # O novo nó se torna o novo fim da fila
            self.fim = novo_no

        # ======================================================================
        # FINALIZAÇÃO (comum aos casos 2 e 3)
        # ======================================================================
        # Incrementa o contador de pacientes (independente do caso)
        self._tamanho += 1

        # Mensagem de confirmação com detalhes do paciente
        status_autismo = "com autismo" if autismo else "sem autismo"
        print(f"-> Paciente {novo_paciente.nome} ({novo_paciente.idade} anos, {status_autismo}) adicionado à fila.")

    # ==========================================================================
    # MÉTODO: atender()
    # ==========================================================================
    # Remove o PRIMEIRO paciente da fila e o retorna.
    # É o método que simula o atendimento propriamente dito.
    # ==========================================================================

    def atender(self):
        """
        Atende (remove) o primeiro paciente da fila.

        FUNCIONAMENTO:
        1. Verifica se a fila está vazia
        2. Se vazia → exibe mensagem de erro e retorna None
        3. Se não vazia → guarda o paciente do início, move o início para o próximo nó
        4. Se a fila ficar vazia, atualiza o 'fim' para None também
        5. Retorna o paciente atendido

        RETORNO:
        - Paciente: o paciente que foi atendido
        - None: se a fila estava vazia
        """

        # --- PASSO 1: Verificar se a fila está vazia ---
        # Se não houver pacientes, não podemos atender ninguém
        if self.esta_vazia():
            print("\n A fila está vazia! Nenhum paciente para atender.")
            return None

        # --- PASSO 2: Guardar o paciente que será atendido ---
        # Precisamos guardar o dado ANTES de remover o nó,
        # senão perderíamos a referência para o paciente
        paciente_atendido = self.inicio.dado

        # --- PASSO 3: Avançar o início da fila ---
        # O novo início será o próximo nó (se existir)
        # Isso efetivamente REMOVE o antigo primeiro nó da fila
        self.inicio = self.inicio.proximo

        # --- PASSO 4: Atualizar o contador ---
        # Um paciente saiu da fila, então diminuímos o contador
        self._tamanho -= 1

        # --- PASSO 5: Verificar se a fila ficou vazia ---
        # Se o novo início é None, significa que não há mais pacientes
        # Nesse caso, o 'fim' também deve ser None (fila vazia)
        if self.inicio is None:
            self.fim = None

        # --- PASSO 6: Mensagem de confirmação e retorno ---
        print(f"\n Atendendo: {paciente_atendido.nome}")
        return paciente_atendido

    # ==========================================================================
    # MÉTODO: listar()
    # ==========================================================================
    # Percorre toda a fila e exibe todos os pacientes na ordem de atendimento.
    # ==========================================================================

    def listar(self):
        """
        Exibe a fila de espera completa.

        FUNCIONAMENTO:
        1. Verifica se a fila está vazia
        2. Se vazia → exibe mensagem informando
        3. Se não vazia → percorre a fila do início ao fim, exibindo cada paciente

        POR QUE PRECISAMOS PERCORRER?
        - A lista encadeada NÃO permite acesso direto a qualquer posição
        - Precisamos começar no 'inicio' e ir seguindo 'proximo' até chegar ao fim

        CUIDADO IMPORTANTE:
        - Usamos 'atual' para percorrer, NÃO modificamos 'inicio'
        - Se modificássemos 'inicio', perderíamos a referência da fila!
        """

        print("\n--- FILA DE ESPERA ATUAL ---")

        # CASO 1: Fila vazia
        if self.esta_vazia():
            print("Nenhum paciente aguardando.")
            print("-------------------------------\n")
            return

        # CASO 2: Fila com pacientes
        # 'atual' começa no início da fila
        atual = self.inicio
        posicao = 1  # Contador para numerar os pacientes na fila

        # Enquanto houver um nó para visitar (atual não é None)
        while atual is not None:
            # Exibe a posição e os dados do paciente
            print(f"{posicao}º -> {atual.dado}")

            # AVANÇA para o próximo nó
            # Esta linha é CRÍTICA! Sem ela, o loop seria infinito.
            # 'atual' recebe a referência do próximo nó armazenada em 'proximo'
            atual = atual.proximo

            # Incrementa a posição para o próximo paciente
            posicao += 1

        # Exibe o total de pacientes na fila (usando o contador interno)
        print(f"Total na fila: {self._tamanho}")
        print("-------------------------------\n")

    # ==========================================================================
    # MÉTODO: tamanho()
    # ==========================================================================
    # Retorna a quantidade atual de pacientes na fila.
    # ==========================================================================

    def tamanho(self) -> int:
        """
        Retorna o número de pacientes aguardando na fila.

        POR QUE USAR UM CONTADOR INTERNO?
        - Percorrer a fila para contar é ineficiente (O(n))
        - Manter um contador atualizado é O(1) - muito mais rápido!
        - O contador é atualizado quando adicionamos (++ ) e quando atendemos (-- )
        """
        return self._tamanho


# ==============================================================================
# PARTE 4: TESTES DO SISTEMA
# ==============================================================================
# Aqui vamos simular o funcionamento da clínica com um cenário real.
# ==============================================================================

# --- PASSO 1: Criar a fila da clínica ---
# Instanciamos um objeto da classe FilaClinica
# Ele começa vazio (inicio = None, fim = None, _tamanho = 0)
clinica = FilaClinica()

# --- PASSO 2: Exibir cabeçalho ---
print("=" * 50)
print("   SISTEMA DE ATENDIMENTO COM PRIORIDADE PARA AUTISMO")
print("=" * 50)

# --- PASSO 3: Simular a chegada de pacientes ---
print("\n=== 1. CHEGADA DE PACIENTES ===")

# Paciente 1: João, 30 anos, SEM autismo
# Vai para o FINAL da fila (primeiro paciente, vira início e fim)
clinica.adicionar("João", 30, autismo=False)

# Paciente 2: Maria, 25 anos, SEM autismo
# Vai para o FINAL da fila (depois do João)
clinica.adicionar("Maria", 25, autismo=False)

# Paciente 3: Carlos, 18 anos, COM autismo (PRIORITÁRIO)
# Deve entrar na FRENTE do João e da Maria
clinica.adicionar("Carlos", 18, autismo=True)

# Paciente 4: Pedro, 40 anos, SEM autismo
# Vai para o FINAL da fila (depois da Maria)
clinica.adicionar("Pedro", 40, autismo=False)

# Paciente 5: Ana, 22 anos, COM autismo (PRIORITÁRIO)
# Deve entrar DEPOIS do Carlos (outro autista) mas ANTES dos normais
clinica.adicionar("Ana", 22, autismo=True)

# Paciente 6: Lucas, 35 anos, SEM autismo
# Vai para o FINAL da fila (depois do Pedro)
clinica.adicionar("Lucas", 35, autismo=False)

# --- PASSO 4: Exibir a fila atual ---
# Deve mostrar: Carlos, Ana, João, Maria, Pedro, Lucas
clinica.listar()

# --- PASSO 5: Simular o atendimento ---
print("\n=== 2. ATENDIMENTO DOS PACIENTES ===")

# Atende o primeiro: deve ser Carlos (prioritário)
clinica.atender()
clinica.listar()  # Mostra a fila após o atendimento

# Atende o segundo: deve ser Ana (prioritário)
clinica.atender()
clinica.listar()  # Mostra a fila após o atendimento

# --- PASSO 6: Atender o restante (ordem normal) ---
print("\n=== 3. ATENDENDO O RESTANTE ===")

# Agora só restam pacientes normais, na ordem de chegada
clinica.atender()   # João
clinica.atender()   # Maria
clinica.atender()   # Pedro
clinica.atender()   # Lucas

# --- PASSO 7: Tentar atender com fila vazia ---
# Deve exibir mensagem de erro
clinica.atender()

# --- PASSO 8: Exibir fila vazia ---
clinica.listar()

   SISTEMA DE ATENDIMENTO COM PRIORIDADE PARA AUTISMO

=== 1. CHEGADA DE PACIENTES ===
-> Paciente João adicionado como o primeiro da fila.
-> Paciente Maria (25 anos, sem autismo) adicionado à fila.
-> Paciente Carlos (18 anos, com autismo) adicionado à fila.
-> Paciente Pedro (40 anos, sem autismo) adicionado à fila.
-> Paciente Ana (22 anos, com autismo) adicionado à fila.
-> Paciente Lucas (35 anos, sem autismo) adicionado à fila.

--- FILA DE ESPERA ATUAL ---
1º -> [Carlos, 18 anos, Autista: Sim - PRIORITÁRIO]
2º -> [Ana, 22 anos, Autista: Sim - PRIORITÁRIO]
3º -> [João, 30 anos, Autista: Não - NORMAL]
4º -> [Maria, 25 anos, Autista: Não - NORMAL]
5º -> [Pedro, 40 anos, Autista: Não - NORMAL]
6º -> [Lucas, 35 anos, Autista: Não - NORMAL]
Total na fila: 6
-------------------------------


=== 2. ATENDIMENTO DOS PACIENTES ===

 Atendendo: Carlos

--- FILA DE ESPERA ATUAL ---
1º -> [Ana, 22 anos, Autista: Sim - PRIORITÁRIO]
2º -> [João, 30 anos, Autista: Não - NORMAL]
3º -> [Maria, 2